# Wayback → Tavily → Classify: dead-company demo harness

**The story:** ~22k of our 44,387 companies are *gone from the live web*, so they were classified on **metadata only** (empty `website_evidence`). This notebook shows we can walk one of those dead companies all the way through the pipeline — its **pre-death Wayback snapshot** → the **exact** Tavily crawl we ran on live companies → the **same** classifier — and then asks the research question:

> Does recovering the dead site's evidence **change** the classification versus the metadata-only label it already has?

**How to use it (inspect-first, one company at a time):**
1. Run every cell top-to-bottom once.
2. Browse the **candidates** table and pick an index `i`.
3. `show(i)` — prints the company + a clickable Wayback link. **Free, no spend.** Open it and judge the footprint (home / product / about / careers).
4. `scrape(i)` — the deliberate paid step: runs Tavily, cleans the evidence, classifies, and shows the **recovered-evidence verdict vs. the stored baseline** side by side.
5. Change `i`, repeat. Use the compare cell to see crawl-vs-extract on the same company.

> Cell 1 is display plumbing (rendering only) — you can skip reading it. The real logic lives in `wayback_machine/tavily_archive_lab.py`, which reuses the live pipeline's crawl config, evidence cleaner, and classifier **unmodified**.

In [ ]:
import html
import json
import pathlib
import sys

from IPython.display import HTML, Markdown, display

# Put the repo root on sys.path so `import wayback_machine...` works no matter
# where the kernel started (Jupyter defaults the cwd to this notebook's folder).
ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from wayback_machine.tavily_archive_lab import ArchiveLab, _rewrite_to_origin, archive_url  # noqa: E402

lab = ArchiveLab()

# ---- display plumbing (rendering only) -----------------------------------
# The 11 classifier output fields, ordered for the baseline-vs-evidence contrast.
KEY_FIELDS = [
    "ai_native", "subclass", "rad_score", "cohort",
    "conf_classification", "conf_rad",
    "reasons_3_points", "sources_used", "verification_critique",
]


def _link(url, label):
    url = str(url or "").strip()
    return f'<a href="{html.escape(url)}" target="_blank">{label}</a>' if url else ""


def render_candidates(df):
    """Index + clickable Wayback links so you can browse and judge substance."""
    head = (
        "<tr><th>i</th><th>name</th><th>captures</th><th>archive span (d)</th>"
        "<th>founded</th><th>alive?</th><th>wayback snapshots</th><th>homepage</th></tr>"
    )
    body = []
    for i, r in df.iterrows():
        links = " · ".join(filter(None, [
            _link(r["target_url"], "pre-death"),
            _link(r["latest_url"], "latest"),
            _link(r["earliest_url"], "earliest"),
        ]))
        body.append(
            f"<tr><td><b>{i}</b></td><td>{html.escape(str(r['name']))}</td>"
            f"<td align=right>{int(r['n_captures']):,}</td>"
            f"<td align=right>{int(r['lifespan_days']):,}</td>"
            f"<td>{html.escape(str(r['founded_date']))}</td>"
            f"<td>{html.escape(str(r['website_alive']))}</td>"
            f"<td>{links}</td>"
            f"<td><code>{html.escape(str(r['homepage_url']))}</code></td></tr>"
        )
    display(HTML(f"<table>{head}{''.join(body)}</table>"))


def show(i):
    """Inspect company `i` for free — prints metadata + a clickable archive link. No spend."""
    c = lab.inspect(i)
    display(Markdown(
        f"### [{i}] {c.name}\n"
        f"- **homepage:** `{c.homepage_url}`\n"
        f"- **founded:** {c.founded_date}  ·  **alive on live web?** `{c.website_alive}`\n"
        f"- **pre-death snapshot:** `{c.closest_ts}`  ·  **last archived:** `{c.death_ts}`\n"
        f"- **archive captures:** {c.n_captures:,}\n\n"
        f"### → [Open the archived site in a new tab]({c.browse_url})\n\n"
        f"_Click through home / product / about / careers to judge the footprint, "
        f"then `scrape({i})` when you're satisfied it's a real, substantial site._"
    ))


def _box(title, text, *, open=False):
    o = " open" if open else ""
    body = html.escape(text or "")
    return (
        f"<details{o}><summary><b>{title}</b></summary>"
        f"<pre style='white-space:pre-wrap;max-height:340px;overflow:auto'>{body}</pre></details>"
    )


def _field_text(row, field):
    if row is None:
        return "—"
    val = row.get(field)
    if val is None or val == "":
        return "—"
    return str(val)


def _contrast_html(res):
    base, ev = res.baseline, res.verdict
    rows = []
    for f in KEY_FIELDS:
        b = _field_text(base, f)
        e = _field_text(ev, f)
        changed = base is not None and ev is not None and b != e
        bg = " style='background:#fff3cd'" if changed else ""
        flag = " ⬅ changed" if changed else ""
        rows.append(
            f"<tr{bg}><th align=left>{f}{flag}</th>"
            f"<td>{html.escape(b)}</td><td>{html.escape(e)}</td></tr>"
        )
    return (
        "<table><tr><th>field</th>"
        "<th>baseline — metadata only (stored)</th>"
        "<th>with recovered evidence (flex)</th></tr>"
        f"{''.join(rows)}</table>"
    )


def _render_scrape(res):
    import json as _json
    c = res.company
    fb = " · _empty-result fallback used_" if res.fallback_used else ""
    display(Markdown(
        f"### Scrape [{c.index}] {c.name} — `method={res.method}` · "
        f"`modifier={res.modifier}` · `scope={res.scope}`{fb}\n"
        f"**Snapshot crawled:** [{res.start_url}]({res.start_url})"
    ))

    # 1) Exact Tavily request(s): endpoint + JSON body. For crawl, an empty-result
    # fallback is shown as a second request when Tavily returned no usable pages.
    display(Markdown("**1. Exact Tavily request payload(s)**"))
    display(HTML(_box(
        "Endpoint + JSON body sent to Tavily",
        _json.dumps(res.tavily_requests, indent=2, ensure_ascii=False),
        open=True,
    )))

    if res.error:
        display(HTML(f"<div style='color:#b00'><b>Tavily error:</b> {html.escape(res.error)}</div>"))
        return

    # 2) Raw API output, before our cleaner touches it.
    display(Markdown("**2. Raw Tavily JSON response**"))
    display(HTML(_box(
        "Raw response returned by Tavily (pre-cleaning)",
        _json.dumps(res.response, indent=2, ensure_ascii=False),
        open=True,
    )))

    d = res.diagnostics
    display(Markdown(
        f"**3. Footprint diagnostics:** {d['pages']} pages · {d['on_domain_pct']}% on-domain "
        f"({d['off_domain_pages']} off-domain) · {d['total_raw_chars']:,} chars · "
        f"{d['credits']} Tavily credits"
    ))
    if d["page_urls"]:
        display(Markdown("**Pages returned (origin URLs after Wayback prefix stripping):**\n" + "\n".join(f"- `{u}`" for u in d["page_urls"])))

    # 4) Exact transformation boundary: raw Tavily response -> origin URLs -> live cleaner.
    display(HTML(_box(
        "Tavily response after archive URL normalization (input to compact_tavily_response)",
        _json.dumps(_rewrite_to_origin(res.response), indent=2, ensure_ascii=False),
    )))
    display(HTML(_box("Recovered website_evidence (output of compact_tavily_response)", res.evidence or "(no usable evidence recovered)", open=True)))
    display(HTML(_box("Exact classifier input (format_user_message)", res.classifier_input)))

    if not res.evidence:
        display(Markdown(
            "**Classification skipped:** Tavily did not return usable `website_evidence`, "
            "so running the classifier would only reproduce a metadata-only verdict. "
            "Try another candidate or compare with `method=\"extract\"`."
        ))
        return

    display(Markdown("**5. Verdict contrast — did recovered evidence change the label?**"))
    display(HTML(_contrast_html(res)))


def scrape(i, **overrides):
    """Deliberate paid step: Tavily-scrape company `i` under CONFIG, classify, contrast."""
    res = lab.scrape(i, **{**CONFIG, **overrides})
    _render_scrape(res)
    return res


print("ready — lab loaded, helpers defined.")

## 1. The problem: survivorship bias

The live cohort was classified from real website evidence (a 5-page Tavily crawl). But for the companies that **died**, the live site is gone — Tavily found nothing, so they were labeled on metadata alone (name, description, categories). That's a survivorship bias: our picture of "what AI-native startups looked like" is skewed toward the ones that survived long enough to be scraped.

The probe in `death_coverage.csv` is finding, for each dead company, a **pre-death Wayback snapshot** we can scrape instead. The candidates below are the resolvable, genuinely-dead companies — ranked by how richly the archive captured them.

In [ ]:
# Dead companies with a genuine pre-death snapshot, richest first.
# Defaults: alive=False (dead only), require_pre_death=True, shared hosts filtered.
# Override to explore: lab.candidates(20, alive=None)  -> include live-but-unscraped too.
cands = lab.candidates(20)
render_candidates(cands)

## 2. The knobs

`CONFIG` is what every `scrape(i)` uses by default. The defaults reproduce the **exact** live crawl. The two archive-specific facts worth demoing:

- **`modifier`** — crawling needs Wayback's `if_` (iframe) mode. With `id_` the archive serves raw bytes and the crawler's links escape to the *dead live domain*; `if_` keeps every link inside the archive. (`None` picks `if_` for crawl, `id_` for extract.)
- **`scope`** — on the archive every page is under `web.archive.org`, so we re-confine the crawl to *this company's* archived pages with a per-company path regex. Turn it off to watch the crawler wander into unrelated archived sites.

Edit a value and re-run a `scrape(i)` cell to see the effect.

In [ ]:
CONFIG = {
    "method": "crawl",    # "crawl" = modern multi-page; "extract" = single-page PIVOT
    "modifier": None,     # None -> if_ for crawl, id_ for extract; force with "if_" / "id_"
    "scope": True,        # confine the crawl to this company's archived pages
    "limit": None,        # None -> modern default (5 pages)
    "max_depth": None,    # None -> modern default (2)
    "instructions": None, # None -> modern page-selection instructions
    "classify": True,     # run the flex classifier on the recovered evidence
}
CONFIG

## 3. Inspect for free — `show(i)`

Set `SELECT` to a row index from the table. `show(SELECT)` prints the company and a clickable Wayback link. **No Tavily call, no spend.** Open the link, click through the archived pages, and confirm it's a real site with substance before scraping.

In [ ]:
SELECT = 1   # <- pick a row index from the candidates table above
show(SELECT)

## 4. Scrape + classify — `scrape(i)`

The deliberate paid step. `scrape(SELECT)` now renders the pipeline as an audit trail, not just a final verdict:

1. **Exact Tavily request payload(s)** — endpoint + JSON body sent to Tavily. If crawl returns no usable pages and the empty-instructions fallback runs, both request bodies are shown.
2. **Raw Tavily JSON response** — the unmodified API response, before our cleaner touches it.
3. **Footprint diagnostics** — page count, on-domain %, raw character count, credits, and the origin URLs after Wayback prefix stripping.
4. **Normalized response into the cleaner** — same response after archive URLs are rewritten back to origin URLs; this is the direct input to `compact_tavily_response`.
5. **Recovered `website_evidence`** — exactly what the cleaner produced, same format the live cohort used.
6. **Classifier input** — the literal `format_user_message` string sent to the model.
7. **Verdict contrast** — only shown when usable evidence exists; otherwise classification is skipped so we do not confuse metadata-only reruns with evidence-based classification.

_(Cached per (company, config), so use `force=True` if you intentionally want to re-spend and refresh a result.)_

In [ ]:
result = scrape(SELECT)

## 5. Compare methodologies (the GO / PIVOT question)

Same company, same snapshot, two ways to scrape it:

- **`crawl`** (GO candidate) — the modern 5-page crawl on the `if_` snapshot, scoped to the company. Matches what the live cohort got.
- **`extract`** (PIVOT fallback) — a single-page `id_` homepage extract. Safer on the archive, but only 1 page, so the comparison to the live cohort is weaker.

If the crawl reliably returns clean, multi-page, on-domain evidence → **GO** (run the 22k through the crawl). If it's noisy or escapes the archive → **PIVOT** to single-page extract and document the 1-vs-5-page gap as a paper limitation.

In [ ]:
# Modern multi-page crawl (the GO candidate)
r_crawl = scrape(SELECT, method="crawl", scope=True)

# Single-page homepage extract (the PIVOT fallback)
r_extract = scrape(SELECT, method="extract")

# Optional: watch the crawler escape the company when scope is off (id_ + no scope)
# r_escape = scrape(SELECT, method="crawl", modifier="id_", scope=False)

## 6. Findings — GO / PIVOT recommendation

_Fill this in after running a handful of companies (pin the verified indices for the demo)._

**Companies verified:** `[ ... ]`  (e.g. indices 0, 3, 5)

**Crawl quality on the archive (`if_` + scope):**
- Pages per company (median): `__`
- On-domain %: `__`
- Evidence reads like the live cohort's? `yes / mostly / no`

**Did recovered evidence change the verdict vs. the metadata-only baseline?**
- `__ / __` companies had a changed `ai_native` / `subclass` / `rad_score`
- Most striking flip: `__`

**Decision:** **GO** (5-page archive crawl for the 22k) / **PIVOT** (single-page extract, document as limitation)

**Why:** `__`